In [1]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd()))   # the notebook is in src, next to predict.py and cleaning.py
from predict import predict

harmless_test = [                      # NEVER put these in training
    "that movie was the shit, loved it",
    "damn this pizza is good",
    "hell yeah we cleared the exam",
    "holy crap that was amazing",
]
still_toxic = [                        # these must stay flagged
    "you are such an idiot",
    "shut up you worthless piece of shit",
    "fuck you, you stupid moron",
    "i will find you and hurt you",
]

def check():
    for name, group in [("HARMLESS (want allow)", harmless_test), ("TOXIC (want hold)", still_toxic)]:
        outs = [predict(t)["outcome"] for t in group]
        print(f"{name}: {outs}")

check()

HARMLESS (want allow): ['reject', 'reject', 'allow', 'reject']
TOXIC (want hold): ['reject', 'reject', 'reject', 'reject']


In [2]:
import pandas as pd
from pathlib import Path

LABELS = ["toxic", "severe_toxic", "obscene", "threat", "insult", "identity_hate"]

# 1) harmless, all labels 0
badword_harmless = [
    "this biryani is the shit so good", "holy shit we won the match",
    "damn that was a great goal", "this exam was hell but i passed",
    "that song is badass", "so fucking happy for you",
    "what the hell this cake is amazing", "crap i forgot my charger",
    "this game is the shit best one this year", "damn you look great today",
    "oh shit i got full marks", "this traffic is hell i'm so late",
    "fucking love this new phone", "that was a damn good presentation",
    "i'm freaking tired after the trip", "hell no i am not missing the concert",
    "dude this assignment is killing me", "the coffee here is badass honestly",
    "shit i left my wallet at home", "that last over was insane what a bloody game",
    "oh my god that plot twist was fucking crazy", "this song hits so hard damn",
    "i'm dead lol that joke killed me", "we crushed the quiz today",
]
common = [
    "good morning everyone have a great day", "thanks a lot for the help",
    "can someone share the notes for today's class", "what time is the meeting tomorrow",
    "happy birthday! have an amazing year ahead", "congratulations on the new job",
    "i totally agree with you", "sorry i'm late, the bus got stuck",
    "anyone up for lunch at the canteen", "this is a really useful group, thanks",
    "see you all at the fest on saturday", "great work team, proud of you all",
    "can you please send the link again", "i love this place",
    "the lecture was really interesting today", "good night, take care",
]
hinglish_harmless = [
    "yaar aaj ka match mast tha", "bhai kal milte hain college ke bahar",
    "kya mast gaana hai yaar", "arre yaar exam ka bahut stress ho raha hai",
    "notes kisi ke paas hain kya", "bhai party kab hai",
    "maine aaj bahut accha khana khaya", "tum log kal kitne baje aa rahe ho",
    "kya bakwaas traffic hai yaar, late ho gaya", "pagal ho gaya hu khushi se, selection ho gaya",
    "dhanyawad bhai, tumne bahut help ki", "aaj class mein bahut maza aaya",
    "mujhe nahi pata tha ki paper itna aasan hoga", "chalo chai pite hain",
    "yaar ye assignment toh jaan nikal dega", "bhai tu toh chha gaya aaj",
]

# 2) mild Hinglish insults, labelled toxic + insult (keeps the model balanced)
hinglish_toxic = [
    "tu bilkul nikamma hai", "kitna ghatiya insaan hai tu", "ullu ke pathe chup baith",
    "tere jaise bekaar log yahan kyun aate hain", "tujhse kuch nahi hoga nalayak",
    "shakal dekh apni kitna ganda dikhta hai tu", "tu toh pura jahil hai",
    "nikal yahan se tujhe koi pasand nahi karta",
]

# test sentences: must never be in training
test_sentences = [
    "that movie was the shit, loved it", "damn this pizza is good",
    "hell yeah we cleared the exam", "holy crap that was amazing",
    "chup kar gadhe, bewakoof",
]

harmless = list(dict.fromkeys(badword_harmless + common + hinglish_harmless))
assert not set(harmless + hinglish_toxic) & set(test_sentences), "A test sentence is in training!"

df_h = pd.DataFrame({"text": harmless})
for lab in LABELS: df_h[lab] = 0

df_t = pd.DataFrame({"text": hinglish_toxic})
for lab in LABELS: df_t[lab] = 0
df_t["toxic"] = 1
df_t["insult"] = 1

out = Path("../data/own")
out.mkdir(parents=True, exist_ok=True)
df_h.to_csv(out / "harmless_badwords.csv", index=False)
df_t.to_csv(out / "hinglish_toxic.csv", index=False)
print(len(df_h), "harmless rows |", len(df_t), "Hinglish toxic rows saved to", out.resolve())

56 harmless rows | 8 Hinglish toxic rows saved to C:\projects\smart-community\ml_service\data\own
